In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [4]:
# Filterning based on Manufacturing or not & Location Recheck
FILE_PATH = "CompaniesList.xlsx"
SHEET_NAME = "Filtering"

reject_keywords = [
    "contract research organization",
    "analytical testing services",
    "clinical trial"
]

manufacturer_keywords = [
    "manufacturing", "manufacture", "plant",
    "facility", "production", "factory",
    "synthesis", "formulation plant"
]

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def get_text_from_url(url):
    try:
        if not url.startswith("http"):
            url = "https://" + url

        response = requests.get(url, headers=HEADERS, timeout=5)
        soup = BeautifulSoup(response.text, "html.parser")

        return soup.get_text(separator=" ").lower()

    except:
        return ""

def get_combined_text(base_url):
    pages = ["", "/about", "/about-us", "/products", "/contact", "/contact-us"]

    full_text = ""

    for p in pages:
        url = base_url.rstrip("/") + p
        text = get_text_from_url(url)
        full_text += " " + text

    return full_text

def classify_company(text):
    if not text:
        return "Flag"

    reject_hits = sum(k in text for k in reject_keywords)
    manufacturer_hits = sum(k in text for k in manufacturer_keywords)

    if reject_hits >= 2 and manufacturer_hits == 0:
        return "Reject"

    if manufacturer_hits >= 2:
        return "Keep"

    return "Flag"

def detect_location(text):
    if not text:
        return "Unknown"

    location_keywords = ["hyderabad", "telangana", "hyd"]

    if any(k in text for k in location_keywords):
        return "Hyderabad"

    return "Unknown"


# MAIN
df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)

results = []
locations = []

for _, row in df.iterrows():
    company = row.get("Company Name", "")
    website = row.get("Website", "")

    if website == "Not found" or pd.isna(website):
        results.append("Reject")
        locations.append("Unknown")
        continue

    text = get_combined_text(website)

    status = classify_company(text)
    location = detect_location(text)

    print(f"{company} → {status}, {location}")

    results.append(status)
    locations.append(location)

df["Layer1_Status"] = results
df["Location"] = locations

df.to_excel("filtered_output.xlsx", index=False)

print("\n Done. Saved as filtered_output.xlsx")

Vishnu Priya Chemicals Pvt Ltd → Keep, Hyderabad
Agrisol (India) Private Limited → Flag, Unknown
Srini Chem → Keep, Hyderabad
Sai Fine Chemicals → Keep, Hyderabad
A.R. Life Sciences Pvt Ltd → Keep, Hyderabad
Synthokem Labs Pvt Ltd → Flag, Unknown
Nakoda Chemicals Ltd → Keep, Hyderabad
Veer Chemie & Aromatics Pvt Ltd → Flag, Unknown
Golden Streak Drugs & Pharma Ltd → Flag, Unknown
Chemit Laboratories → Flag, Unknown
Basr Fine Chemicals Pvt Ltd → Flag, Unknown
Syn Finechem Laboratories Pvt Ltd → Flag, Unknown
G M Fine Chemicals → Flag, Unknown
Bondbay Pharmaceuticals Pvt Ltd → Keep, Hyderabad
Dinakara Life Sciences Pvt Ltd → Keep, Hyderabad
Novasyn Organics Pvt Ltd → Keep, Hyderabad
MYK Arment Private Limited → Keep, Hyderabad
Prathista Industries Ltd → Keep, Unknown
Tyche Industries Ltd → Flag, Unknown
Espi Industries & Chemicals Pvt Ltd → Keep, Hyderabad
Vasudha Pharma Chem Limited → Keep, Hyderabad
Sravani Life Sciences → Keep, Hyderabad
Sri Krishna Pharma → Keep, Hyderabad
Kreative O

In [5]:
# Removing Service based companies

# -----------------------------
# CONFIG
# -----------------------------
INPUT_FILE = "filtered_output.xlsx"
OUTPUT_FILE = "final_sanity_filtered.xlsx"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

service_keywords = [
    "contract research", "cro", "analytical services",
    "testing services", "clinical research", "lab services"
]

generic_pharma_keywords = [
    "bulk drugs", "formulations", "finished dosage",
    "tablets", "capsules", "injectables", "api manufacturer"
]

manufacturer_keywords = [
    "manufacturing", "manufacture", "plant",
    "facility", "production", "factory",
    "synthesis"
]

# -----------------------------
# FUNCTIONS
# -----------------------------
def get_text_from_url(url):
    try:
        if not url.startswith("http"):
            url = "https://" + url

        response = requests.get(url, headers=HEADERS, timeout=5)
        soup = BeautifulSoup(response.text, "html.parser")

        return soup.get_text(separator=" ").lower()

    except:
        return ""

def get_combined_text(base_url):
    pages = ["", "/about", "/about-us", "/products", "/contact", "/contact-us"]

    full_text = ""

    for p in pages:
        url = base_url.rstrip("/") + p
        text = get_text_from_url(url)
        full_text += " " + text

    return full_text

def sanity_filter(text):
    if not text:
        return "Flag"

    service_hits = sum(k in text for k in service_keywords)
    pharma_hits = sum(k in text for k in generic_pharma_keywords)
    manufacturer_hits = sum(k in text for k in manufacturer_keywords)

    if service_hits >= 2 and manufacturer_hits == 0:
        return "Reject_Service"

    if pharma_hits >= 2:
        return "Reject_Pharma"

    return "Pass"

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_excel(INPUT_FILE)

# -----------------------------
# FILTER (recommended)
# -----------------------------
filtered_df = df[
    (df["Layer1_Status"] == "Keep") &
    (df["Location"] == "Hyderabad")
].copy()

# If you want ALL rows instead, comment above and use:
# filtered_df = df.copy()

# -----------------------------
# PROCESS
# -----------------------------
sanity_results = []

for _, row in filtered_df.iterrows():
    company = row.get("Company Name", "")
    website = row.get("Website", "")

    if website == "Not found" or pd.isna(website):
        sanity_results.append("Reject")
        continue

    text = get_combined_text(website)

    result = sanity_filter(text)

    print(f"{company} → {result}")
    sanity_results.append(result)

filtered_df["Sanity_Filter"] = sanity_results

# -----------------------------
# SAVE OUTPUT
# -----------------------------
filtered_df.to_excel(OUTPUT_FILE, index=False)

print("\n✅ Done. Saved as final_sanity_filtered.xlsx")

Vishnu Priya Chemicals Pvt Ltd → Pass
Srini Chem → Pass
Sai Fine Chemicals → Pass
A.R. Life Sciences Pvt Ltd → Pass
Nakoda Chemicals Ltd → Pass
Bondbay Pharmaceuticals Pvt Ltd → Pass
Dinakara Life Sciences Pvt Ltd → Pass
Novasyn Organics Pvt Ltd → Pass
MYK Arment Private Limited → Pass
Prathista Industries Ltd → Pass
Espi Industries & Chemicals Pvt Ltd → Pass
Vasudha Pharma Chem Limited → Pass
Sravani Life Sciences → Pass
Sri Krishna Pharma → Reject_Pharma
Kreative Organics Pvt Ltd → Pass
Shodhana Laboratories Ltd → Pass
Sychar Chemicals → Pass
Shobha Life Sciences → Pass
Sris Synthesis Pvt Ltd → Pass
Lucent Drugs → Pass
Sapala Organics Pvt Ltd → Pass
Kekule Pharma → Pass
Optimus Pharma → Reject_Pharma
Hygro Chemicals Pharmtek Pvt Ltd → Pass
Sain Industries Pvt Ltd → Pass
Covalent Laboratories → Pass
Stereokem Pvt Ltd → Pass
Suvvi Chemical → Pass

✅ Done. Saved as final_sanity_filtered.xlsx
